# Comparing Different Decoder Architectures

This notebook demonstrates how to use different decoder architectures (Autoencoder, VAE, GAN, Diffusion) with the SMOTE for Images pipeline and compare their performance.

## Supported Decoder Types

1. **Autoencoder**: Simple reconstruction-based decoder
2. **VAE**: Variational Autoencoder with probabilistic latent space
3. **GAN**: Generative Adversarial Network decoder
4. **Diffusion**: Diffusion model decoder (advanced)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import time

# Import all decoder types
from smote_image_synthesis.decoders.autoencoder_decoder import AutoencoderDecoder
from smote_image_synthesis.decoders.vae_decoder import VAEDecoder
from smote_image_synthesis.decoders.gan_decoder import GANDecoder
from smote_image_synthesis.decoders.diffusion_decoder import DiffusionDecoder

# Import trainers
from smote_image_synthesis.decoders.autoencoder_trainer import AutoencoderTrainer
from smote_image_synthesis.decoders.vae_trainer import VAETrainer
from smote_image_synthesis.decoders.gan_trainer import GANTrainer
from smote_image_synthesis.decoders.diffusion_trainer import DiffusionTrainer

# Import other components
from smote_image_synthesis.encoders.resnet_encoder import ResNetEncoder
from smote_image_synthesis.quality.assessor import QualityAssessor
from smote_image_synthesis.pipeline import SynthesisPipeline

## 1. Prepare Data and Encoder

Set up common components that will be used with all decoder types.

In [ ]:
# Generate sample data
def create_structured_data(n_samples=200, image_size=(3, 64, 64)):
    """
    Create more structured sample data for better decoder comparison.
    """
    images = []
    labels = []
    
    for class_id in range(3):
        n_class = n_samples // 3 if class_id < 2 else n_samples - 2 * (n_samples // 3)
        
        for i in range(n_class):
            # Create class-specific patterns
            img = torch.randn(*image_size) * 0.1
            
            if class_id == 0:  # Horizontal stripes
                for j in range(0, image_size[1], 8):
                    img[:, j:j+4, :] += 0.5
            elif class_id == 1:  # Vertical stripes
                for j in range(0, image_size[2], 8):
                    img[:, :, j:j+4] += 0.5
            else:  # Checkerboard
                for j in range(0, image_size[1], 8):
                    for k in range(0, image_size[2], 8):
                        if (j // 8 + k // 8) % 2 == 0:
                            img[:, j:j+8, k:k+8] += 0.5
            
            images.append(torch.clamp(img, 0, 1))
            labels.append(class_id)
    
    return torch.stack(images), np.array(labels)

# Create dataset
images, labels = create_structured_data(n_samples=150)
print(f"Created dataset: {images.shape}, classes: {np.bincount(labels)}")

# Split data
train_size = int(0.8 * len(images))
train_images, val_images = images[:train_size], images[train_size:]
train_labels, val_labels = labels[:train_size], labels[train_size:]

# Create shared encoder
encoder = ResNetEncoder(
    architecture='resnet18',
    embedding_dim=256,
    pretrained=True,
    freeze_backbone=True
)

# Encode images once for all decoders
print("Encoding images...")
with torch.no_grad():
    train_embeddings = encoder.encode_batch(train_images)
    val_embeddings = encoder.encode_batch(val_images)

print(f"Embeddings shape: {train_embeddings.shape}")

## 2. Autoencoder Decoder

Simple reconstruction-based decoder with MSE loss.

In [ ]:
# Create and train Autoencoder
print("=== Autoencoder Decoder ===")

autoencoder = AutoencoderDecoder(
    embedding_dim=256,
    image_shape=(3, 64, 64),
    hidden_dims=[512, 256, 128],
    use_batch_norm=True,
    dropout_rate=0.1
)

# Train autoencoder
ae_trainer = AutoencoderTrainer(
    decoder=autoencoder,
    learning_rate=0.001,
    use_perceptual_loss=True,
    perceptual_weight=0.1
)

start_time = time.time()
ae_history = ae_trainer.train(
    train_embeddings=train_embeddings,
    train_images=train_images,
    val_embeddings=val_embeddings,
    val_images=val_images,
    num_epochs=20,
    batch_size=16
)
ae_train_time = time.time() - start_time

print(f"Autoencoder training completed in {ae_train_time:.2f}s")
print(f"Final loss: {ae_history['val_loss'][-1]:.4f}")

## 3. VAE Decoder

Variational Autoencoder with probabilistic latent space and KL divergence regularization.

In [ ]:
# Create and train VAE
print("\n=== VAE Decoder ===")

vae = VAEDecoder(
    embedding_dim=256,
    latent_dim=128,
    image_shape=(3, 64, 64),
    hidden_dims=[512, 256, 128],
    use_batch_norm=True
)

# Train VAE
vae_trainer = VAETrainer(
    vae_decoder=vae,
    learning_rate=0.001,
    beta=1.0,  # KL weight
    use_perceptual_loss=True
)

start_time = time.time()
vae_history = vae_trainer.train(
    train_embeddings=train_embeddings,
    train_images=train_images,
    val_embeddings=val_embeddings,
    val_images=val_images,
    num_epochs=20,
    batch_size=16
)
vae_train_time = time.time() - start_time

print(f"VAE training completed in {vae_train_time:.2f}s")
print(f"Final reconstruction loss: {vae_history['val_recon_loss'][-1]:.4f}")
print(f"Final KL loss: {vae_history['val_kl_loss'][-1]:.4f}")

## 4. GAN Decoder

Generative Adversarial Network decoder with adversarial training.

In [ ]:
# Create and train GAN
print("\n=== GAN Decoder ===")

gan = GANDecoder(
    embedding_dim=256,
    image_shape=(3, 64, 64),
    hidden_dims=[512, 256, 128],
    use_spectral_norm=True,
    use_self_attention=False  # Disable for faster training
)

# Train GAN
gan_trainer = GANTrainer(
    gan_decoder=gan,
    generator_lr=0.0002,
    discriminator_lr=0.0002,
    use_feature_matching=True,
    feature_matching_weight=10.0
)

start_time = time.time()
gan_history = gan_trainer.train(
    train_embeddings=train_embeddings,
    train_images=train_images,
    val_embeddings=val_embeddings,
    val_images=val_images,
    num_epochs=15,  # Fewer epochs for GAN
    batch_size=16
)
gan_train_time = time.time() - start_time

print(f"GAN training completed in {gan_train_time:.2f}s")
print(f"Final generator loss: {gan_history['val_g_loss'][-1]:.4f}")
print(f"Final discriminator loss: {gan_history['val_d_loss'][-1]:.4f}")

## 5. Diffusion Decoder (Optional)

Advanced diffusion model decoder. This is computationally intensive and optional.

In [ ]:
# Create and train Diffusion model (optional - comment out if too slow)
print("\n=== Diffusion Decoder (Optional) ===")

# Uncomment to train diffusion model
# diffusion = DiffusionDecoder(
#     embedding_dim=256,
#     image_shape=(3, 64, 64),
#     num_timesteps=100,  # Reduced for faster training
#     use_attention=False
# )

# diffusion_trainer = DiffusionTrainer(
#     diffusion_decoder=diffusion,
#     learning_rate=0.0001
# )

# start_time = time.time()
# diffusion_history = diffusion_trainer.train(
#     train_embeddings=train_embeddings,
#     train_images=train_images,
#     val_embeddings=val_embeddings,
#     val_images=val_images,
#     num_epochs=10,
#     batch_size=8
# )
# diffusion_train_time = time.time() - start_time

print("Diffusion model training skipped for demo (uncomment to enable)")
diffusion = None
diffusion_train_time = 0

## 6. Generate and Compare Outputs

Generate images using each decoder and compare quality.

In [ ]:
# Generate images with each decoder
test_embeddings = val_embeddings[:8]  # Use first 8 validation embeddings
test_images = val_images[:8]

# Generate with each decoder
with torch.no_grad():
    ae_generated = autoencoder.decode_batch(test_embeddings)
    vae_generated = vae.decode_batch(test_embeddings)
    gan_generated = gan.decode_batch(test_embeddings)
    
    if diffusion is not None:
        diffusion_generated = diffusion.decode_batch(test_embeddings)
    else:
        diffusion_generated = None

# Plot comparison
def plot_decoder_comparison(original, ae_gen, vae_gen, gan_gen, diff_gen=None, n_samples=4):
    n_rows = 5 if diff_gen is not None else 4
    fig, axes = plt.subplots(n_rows, n_samples, figsize=(16, n_rows * 3))
    
    row_labels = ['Original', 'Autoencoder', 'VAE', 'GAN']
    if diff_gen is not None:
        row_labels.append('Diffusion')
    
    generations = [original, ae_gen, vae_gen, gan_gen]
    if diff_gen is not None:
        generations.append(diff_gen)
    
    for row, (label, imgs) in enumerate(zip(row_labels, generations)):
        for col in range(n_samples):
            if col < len(imgs):
                img = imgs[col].permute(1, 2, 0).cpu().numpy()
                axes[row, col].imshow(np.clip(img, 0, 1))
            
            if col == 0:
                axes[row, col].set_ylabel(label, fontsize=12)
            axes[row, col].set_xticks([])
            axes[row, col].set_yticks([])
    
    plt.tight_layout()
    plt.show()

plot_decoder_comparison(
    test_images, ae_generated, vae_generated, gan_generated, diffusion_generated
)

## 7. Quantitative Quality Assessment

Compare decoders using quality metrics.

In [ ]:
# Quality assessment
quality_assessor = QualityAssessor(
    metrics=['mse', 'ssim', 'psnr', 'mae'],
    compute_diversity=True
)

# Evaluate each decoder
decoders = {
    'Autoencoder': ae_generated,
    'VAE': vae_generated,
    'GAN': gan_generated
}

if diffusion_generated is not None:
    decoders['Diffusion'] = diffusion_generated

results = {}
for name, generated in decoders.items():
    print(f"\nEvaluating {name}...")
    result = quality_assessor.evaluate_quality(
        synthetic_images=generated,
        real_images=test_images,
        return_detailed=True
    )
    results[name] = result
    
    # Print metrics
    for metric, value in result['metrics'].items():
        print(f"  {metric.upper()}: {value:.4f}")

## 8. Performance Comparison

Compare training time and quality metrics across decoders.

In [ ]:
# Create comparison table
import pandas as pd

comparison_data = []
train_times = {
    'Autoencoder': ae_train_time,
    'VAE': vae_train_time,
    'GAN': gan_train_time,
    'Diffusion': diffusion_train_time
}

for name in decoders.keys():
    row = {'Decoder': name, 'Training Time (s)': train_times[name]}
    
    # Add quality metrics
    for metric, value in results[name]['metrics'].items():
        row[metric.upper()] = value
    
    # Add diversity metrics
    if 'diversity' in results[name]:
        row['Diversity Index'] = results[name]['diversity'].get('diversity_index', 0)
    
    comparison_data.append(row)

df = pd.DataFrame(comparison_data)
print("\nDecoder Comparison:")
print("=" * 80)
print(df.round(4))

# Plot metrics comparison
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

metrics_to_plot = ['MSE', 'SSIM', 'PSNR', 'Training Time (s)']
for i, metric in enumerate(metrics_to_plot):
    if metric in df.columns:
        axes[i].bar(df['Decoder'], df[metric])
        axes[i].set_title(metric)
        axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 9. Training Loss Visualization

Compare training curves for different decoders.

In [ ]:
# Plot training curves
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Autoencoder loss
axes[0, 0].plot(ae_history['train_loss'], label='Train')
axes[0, 0].plot(ae_history['val_loss'], label='Validation')
axes[0, 0].set_title('Autoencoder Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True)

# VAE losses
axes[0, 1].plot(vae_history['train_total_loss'], label='Total Loss')
axes[0, 1].plot(vae_history['train_recon_loss'], label='Reconstruction')
axes[0, 1].plot(vae_history['train_kl_loss'], label='KL Divergence')
axes[0, 1].set_title('VAE Losses')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend()
axes[0, 1].grid(True)

# GAN losses
axes[1, 0].plot(gan_history['train_g_loss'], label='Generator')
axes[1, 0].plot(gan_history['train_d_loss'], label='Discriminator')
axes[1, 0].set_title('GAN Losses')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].legend()
axes[1, 0].grid(True)

# Comparison of final losses
final_losses = {
    'Autoencoder': ae_history['val_loss'][-1],
    'VAE': vae_history['val_total_loss'][-1],
    'GAN': gan_history['val_g_loss'][-1]
}

axes[1, 1].bar(final_losses.keys(), final_losses.values())
axes[1, 1].set_title('Final Validation Loss')
axes[1, 1].set_ylabel('Loss')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 10. Recommendations and Use Cases

Based on the comparison, here are recommendations for different scenarios:

In [ ]:
# Generate recommendations based on results
def generate_recommendations(results, train_times):
    recommendations = []
    
    # Find best performer for each metric
    best_mse = min(results.items(), key=lambda x: x[1]['metrics']['mse'])
    best_ssim = max(results.items(), key=lambda x: x[1]['metrics']['ssim'])
    fastest = min(train_times.items(), key=lambda x: x[1] if x[1] > 0 else float('inf'))
    
    print("=== DECODER RECOMMENDATIONS ===")
    print()
    
    print("🏆 BEST QUALITY (MSE):", best_mse[0])
    print(f"   MSE: {best_mse[1]['metrics']['mse']:.4f}")
    print()
    
    print("🏆 BEST PERCEPTUAL QUALITY (SSIM):", best_ssim[0])
    print(f"   SSIM: {best_ssim[1]['metrics']['ssim']:.4f}")
    print()
    
    print("⚡ FASTEST TRAINING:", fastest[0])
    print(f"   Time: {fastest[1]:.2f}s")
    print()
    
    print("📋 USE CASE RECOMMENDATIONS:")
    print()
    print("• AUTOENCODER:")
    print("  - Best for: Quick prototyping, simple reconstruction tasks")
    print("  - Pros: Fast training, stable, good baseline")
    print("  - Cons: May lack diversity, deterministic output")
    print()
    
    print("• VAE:")
    print("  - Best for: Probabilistic generation, latent space exploration")
    print("  - Pros: Smooth latent space, good for interpolation")
    print("  - Cons: May produce blurry images, KL collapse risk")
    print()
    
    print("• GAN:")
    print("  - Best for: High-quality, sharp image generation")
    print("  - Pros: Sharp details, realistic textures")
    print("  - Cons: Training instability, mode collapse risk")
    print()
    
    print("• DIFFUSION:")
    print("  - Best for: State-of-the-art quality, diverse generation")
    print("  - Pros: Excellent quality, stable training")
    print("  - Cons: Very slow inference, computationally expensive")

generate_recommendations(results, train_times)

## Conclusion

This notebook demonstrated how to:

1. **Configure different decoder architectures** for the SMOTE pipeline
2. **Train each decoder type** with appropriate hyperparameters
3. **Compare quality metrics** across different architectures
4. **Analyze training efficiency** and convergence behavior
5. **Generate recommendations** based on use case requirements

### Key Takeaways

- **Autoencoder**: Best for quick prototyping and stable baseline results
- **VAE**: Ideal for smooth latent space and probabilistic generation
- **GAN**: Produces sharp, realistic images but requires careful tuning
- **Diffusion**: State-of-the-art quality but computationally expensive

### Next Steps

- Experiment with different hyperparameters for each decoder
- Try hybrid approaches combining multiple decoders
- Test on real datasets with different characteristics
- Implement custom decoder architectures for specific domains